# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [16]:
import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

for path in ['sql', 'reports', 'screenshots']:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv('QBC12_DB_HOST', '185.50.38.163')    # this is in the excel file give in Quera
DB_PORT = os.getenv('QBC12_DB_PORT', '32112')
DB_NAME = os.getenv('QBC12_DB_NAME', 'qbc12_airbnb')
DB_USER = os.getenv('QBC12_DB_USER', 'student_hamed_hamzeh') or input('DB user: ').strip()
DB_PASSWORD = os.getenv('QBC12_DB_PASSWORD', 'XmKTvFl87gMaun3Uuo') or input('DB password: ').strip()
STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or DB_USER.replace('student_', '')

safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
STUDENT_SCHEMA = f'student_{safe_student}'
engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}', pool_pre_ping=True)
with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text('select version()')).scalar()
STUDENT_SCHEMA, version[:80]

('student_hamed_hamzeh',
 'PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by g')

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [17]:
columns_sql = '''
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
'''
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [18]:
row_count_sql = '''
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
'''
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


Searching for and inspecting the neighbourhood table

In [19]:
pd.read_sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='core'
ORDER BY table_name;
""", engine)

,table_name
0,calendar_day
1,host
2,listing
3,neighbourhood
4,review


In [20]:
pd.read_sql("""
SELECT *
FROM core.neighbourhood
LIMIT 5;
""", engine)

,neighbourhood_id,name
0,1,Centrum-Oost
1,2,Centrum-West
2,3,Oostelijk Havengebied - Indische Buurt
3,4,Westerpark
4,5,Slotervaart


## 2. Create your sandbox schema

This is the only place you write database objects.

In [21]:
pd.read_sql("""
SELECT schema_name
FROM information_schema.schemata
ORDER BY schema_name;
""", engine)

,schema_name
0,core
1,information_schema
2,pg_catalog
3,public
4,student_hamed_hamzeh


In [22]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

pd.read_sql(f"""
SELECT schema_name
FROM information_schema.schemata
WHERE schema_name = '{STUDENT_SCHEMA}';
""", engine)

,schema_name
0,student_hamed_hamzeh


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

In [23]:
pd.read_sql("""
SELECT min(date), max(date)
FROM core.calendar_day;
""", engine)

,min,max
0,2025-09-11,2026-09-10


In [24]:
pd.read_sql("""
SELECT
    listing_id,
    min(date) as first_date,
    max(date) as last_date,
    count(*) as num_days
FROM core.calendar_day
GROUP BY listing_id
LIMIT 5;
""", engine)

,listing_id,first_date,last_date,num_days
0,27886,2025-09-11,2026-09-10,365
1,28871,2025-09-11,2026-09-10,365
2,29051,2025-09-11,2026-09-10,365
3,44391,2025-09-11,2026-09-10,365
4,48373,2025-09-11,2026-09-10,365


In [25]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = '''
SELECT
    listing_id,
    avg(price) as avg_calendar_price_30,
    avg(available::int) as availability_30_rate
FROM core.calendar_day
WHERE date BETWEEN
    (select min(date) from core.calendar_day)
    and (select min(date) from core.calendar_day) + interval '30 days'
GROUP BY listing_id;
'''

pd.read_sql(calendar_30_sql, engine)

,listing_id,avg_calendar_price_30,availability_30_rate
0,1443670960781261954,None,0.000000
1,896043282611946316,None,0.000000
2,958726726744532841,None,0.032258
3,39969190,None,0.000000
4,1476051889347548382,None,0.193548
...,...,...,...
10475,3225605,None,0.064516
10476,30251394,None,0.000000
10477,13449471,None,0.000000
10478,898657507954178190,None,0.000000


In [26]:
pd.read_sql("""
SELECT
    count(*) as total,
    count(price) as non_null_price
FROM core.calendar_day;
""", engine)

,total,non_null_price
0,3825200,0


In [27]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = '''
SELECT
    listing_id,
    count(*) as total_reviews
FROM core.review
GROUP BY listing_id
'''

pd.read_sql(review_counts_sql, engine)

,listing_id,total_reviews
0,1443670960781261954,5
1,896043282611946316,2
2,39969190,10
3,958726726744532841,23
4,1093563123501570178,4
...,...,...
9378,42197883,3
9379,3225605,52
9380,30251394,13
9381,13449471,10


In [28]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

baseline_sql = '''
WITH calendar_30 AS (
    SELECT
        listing_id,
        avg(price) as avg_calendar_price_30,
        avg(available::int) as availability_30_rate
    FROM core.calendar_day
    WHERE date BETWEEN
        (select min(date) from core.calendar_day)
        and (select min(date) from core.calendar_day) + interval '30 days'
    GROUP BY listing_id
),

review_counts AS (
    SELECT
        listing_id,
        count(*) as total_reviews
    FROM core.review
    GROUP BY listing_id
),

listing_base as (
    SELECT
        listing_id,
        neighbourhood_id,
        listing_price,
        minimum_nights
    FROM core.listing
),

features as (
    SELECT
        l.listing_id,
        l.neighbourhood_id,
        l.listing_price,
        l.minimum_nights,
        c.avg_calendar_price_30,
        c.availability_30_rate,
        r.total_reviews
    FROM listing_base l
    LEFT JOIN calendar_30 c ON l.listing_id = c.listing_id
    LEFT JOIN review_counts r ON l.listing_id = r.listing_id
),

with_neighbourhood as (
    SELECT
        f.*,
        n.name as neighbourhood
    FROM features f
    LEFT JOIN core.neighbourhood n
        ON f.neighbourhood_id = n.neighbourhood_id
)

SELECT
    neighbourhood,
    count(*) as num_listings,
    avg(listing_price) as avg_price,
    percentile_cont(0.5) within group (order by listing_price) as median_price,
    sum(total_reviews) as total_reviews,
    sum(total_reviews)::float / count(*) as reviews_per_listing,
    avg(availability_30_rate) as availability_30_rate
FROM with_neighbourhood
GROUP BY neighbourhood;
'''

Path('sql/01_baseline_neighbourhood_summary.sql').write_text(baseline_sql)

1567

In [29]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times

baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(            neighbourhood  num_listings    avg_price  median_price  \
 0         Bijlmer-Centrum            66   144.514286         134.0   
 1            Bijlmer-Oost            38   157.363636         127.5   
 2           Bos en Lommer           547   204.674330         184.0   
 3  Buitenveldert - Zuidas           125  3013.219512         182.5   
 4            Centrum-Oost           923   307.724138         240.0   
 
    total_reviews  reviews_per_listing  availability_30_rate  
 0         6579.0            99.681818              0.334311  
 1         1383.0            36.394737              0.185908  
 2        16461.0            30.093236              0.128678  
 3         3655.0            29.240000              0.265290  
 4        76899.0            83.314193              0.222312  ,
 [0.6156339999870397, 0.5414075999869965, 0.5491531999432482])

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [ ]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

# Write your code here.

In [ ]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

# Write your code here.

## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [ ]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f'''
-- write SQL here
'''
Path('sql/02_create_materialized_view.sql').write_text(optimized_sql)

In [ ]:
# TODO 5.2
# Execute optimized_sql statement by statement.

# Write your code here.

In [ ]:
check_sql = f'''select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;'''
pd.read_sql(check_sql, engine)

## 6. Compare latency

Numbers or it did not happen.

In [ ]:
dashboard_sql = f'''
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate, availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame([
    {'query': 'baseline_direct_query', 'best_seconds': min(baseline_times), 'avg_seconds': sum(baseline_times)/len(baseline_times)},
    {'query': 'materialized_view_read', 'best_seconds': min(dashboard_times), 'avg_seconds': sum(dashboard_times)/len(dashboard_times)},
])
perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']
perf

## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [ ]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

# Write your code here.

In [ ]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```